# Class-conditional conformal prediction with data augmentation:  robustness to the classification algorithm

Four configurations, one question:

> At the same data-generating process and classwise calibration, is the
> prediction-set reduction from orbit averaging also observed with classifiers
> other than XGBoost?

This is **not** a comparison of which classifier is better. The relevant
comparisons stay within each classifier, among Classwise APS, Classwise TTA-Avg
and Classwise orbit-averaged APS.

| | |
|---|---|
| classifiers | Random Forest, MLP |
| calibration | `n_cal` = 250, 1250, i.e. E[N_rare] = 10, 50 |
| alpha | 0.10 |
| training augmentation | off |
| everything else | identical to the main experiment |

**Why 10 and 50.** Ten sits just above the minimum finite-threshold
requirement, so a non-negligible fraction of replications still degenerates;
fifty is the stable operating regime, where the comparison is about efficiency
rather than infinite thresholds.

**Why training augmentation is off.** It was analysed in the main experiment
through a dedicated ablation. Including it here would double the cost and mix
two questions — robustness to the classifier, and its interaction with training
augmentation. Only the first is asked here.

**Why alpha = 0.05 is not crossed in.** The main experiment already shows the
qualitative result is robust to the nominal level. Crossing classifier with
alpha would add cost without adding a question.

XGBoost is not re-run: its rows come from the main results directory and serve
as the reference in the final table.

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

BASE         = "/content/drive/MyDrive/Conformal_Prediction_Research/Class_Conditional_CP_with_Data_Augmentation/csda_revision"
CODE_DIR     = f"{BASE}/code"
MAIN_DIR     = f"{BASE}/results_R300"          # XGBoost, read only
RESULTS_DIR  = f"{BASE}/results_robust_clf"    # this notebook writes here

import sys, os
sys.path.insert(0, CODE_DIR)
os.makedirs(RESULTS_DIR, exist_ok=True)

!pip install -q xgboost pyarrow 2>/dev/null | tail -1

import numpy, pandas, sklearn, xgboost
print("numpy", numpy.__version__, "| pandas", pandas.__version__,
      "| sklearn", sklearn.__version__, "| xgboost", xgboost.__version__)
print("main results ->", MAIN_DIR)
print("this run     ->", RESULTS_DIR)

Mounted at /content/drive
numpy 2.0.2 | pandas 2.2.2 | sklearn 1.6.1 | xgboost 3.3.0
main results -> /content/drive/MyDrive/Conformal_Prediction_Research/Class_Conditional_CP_with_Data_Augmentation/csda_revision/results_R300
this run     -> /content/drive/MyDrive/Conformal_Prediction_Research/Class_Conditional_CP_with_Data_Augmentation/csda_revision/results_robust_clf


In [ ]:
# The modules on Drive must be current: the MLP arm needs the standardisation
# pipeline, without which the network is fitted on unscaled features.
import inspect, simulate as S, calibration as C, metrics as M

checks = {
    "simulate: checkpointing":     "checkpoint_every" in inspect.signature(S.run).parameters,
    "simulate: separate eps_tr":   "rng_gtr" in inspect.getsource(S.run_replication),
    "simulate: MLP standardised":  "StandardScaler" in inspect.getsource(S.build_model),
    "calibration: warning filter": "empty slice" in inspect.getsource(C._silent),
    "metrics: load skips ckpt":    "__ckpt" in inspect.getsource(M.load),
    "metrics: classifier table":   hasattr(M, "classifier_robustness"),
}
for k, v in checks.items():
    print(("  ok  " if v else " STALE") + " | " + k)
if not all(checks.values()):
    raise SystemExit("re-upload the stale modules to Drive, then restart the runtime")

  ok   | simulate: checkpointing
  ok   | simulate: separate eps_tr
  ok   | simulate: MLP standardised
  ok   | calibration: warning filter
  ok   | metrics: load skips ckpt
  ok   | metrics: classifier table


## 2. Configuration

Identical to the main experiment except for the classifier, the calibration
size, and training augmentation being off.

The MLP is fitted inside a pipeline whose first step is standardisation, so the
scaler is estimated on the training sample only and applied unchanged to
calibration and test. Group transformations act on the raw features, before
scaling, so exact invariance is preserved. Hyperparameters are library defaults,
fixed in advance and identical across replications; they are not selected on the
basis of conformal results.

In [ ]:
from simulate import SimConfig, run, run_replication, _population_size
from dataclasses import replace
import time

BASE_CFG = SimConfig(
    scenario="s3_nuisance",
    n_informative=5, n_nuisance=20, class_sep=0.8, balanced=False,
    n_train=4000, n_test_pop=2000, n_test_balanced_per_class=500,
    n_transforms=16, alpha=0.10,
    run_train_aug=False,          # off: this notebook asks one question only
    strict=True,
    outdir=RESULTS_DIR,
)

R      = 300
MODELS = ["RandomForest", "MLP"]
NCAL   = [250, 1250]              # E[N_rare] = 10, 50

todo = [(m, n) for m in MODELS for n in NCAL]
for m, n in todo:
    c = replace(BASE_CFG, model=m, n_cal=n)
    print(f"{m:14s} n_cal={n:>5}  E[N_rare]={c.expected_rare_cal():>4.0f}  "
          f"population={_population_size(c):>8,}")
print(f"\n{len(todo)} configurations at R={R}")

RandomForest   n_cal=  250  E[N_rare]=  10  population=  37,500
RandomForest   n_cal= 1250  E[N_rare]=  50  population=  39,500
MLP            n_cal=  250  E[N_rare]=  10  population=  37,500
MLP            n_cal= 1250  E[N_rare]=  50  population=  39,500

4 configurations at R=300


## 3. Smoke test

One replication per configuration, with the per-replication assertions on.
Checks: MLP convergence, every class present in training, `predict_proba` rows
summing to one, no missing values, and the cost per replication.

A `ConvergenceWarning` here is worth knowing about. `max_iter` is fixed in
advance and would not be raised in reaction to the results, but an MLP that
does not converge should be reported rather than silently accepted.

In [ ]:
import warnings, pandas as pd
from sklearn.exceptions import ConvergenceWarning

timings = {}
for model, n_cal in todo:
    cfg = replace(BASE_CFG, model=model, n_cal=n_cal, n_reps=1)
    with warnings.catch_warnings(record=True) as w:
        warnings.simplefilter("always")
        t = time.time(); rows = run_replication(cfg, 0); dt = time.time() - t
        conv = [x for x in w if issubclass(x.category, ConvergenceWarning)]
    d = pd.DataFrame(rows)
    timings[(model, n_cal)] = dt
    print(f"{model:14s} n_cal={n_cal:>5} | {dt:5.1f}s | {len(d):>3} rows | "
          f"methods={d.method.nunique()} | train_aug={sorted(d.train_aug.unique())} | "
          f"NaN={int(d.isna().sum().sum())} | convergence warnings={len(conv)}")

total = sum(timings.values()) * R / 3600
print(f"\nprojected total at R={R}: {total:.1f} h")
for (m, n), dt in timings.items():
    print(f"   {m:14s} n_cal={n:>5}: {dt*R/60:5.0f} min")

RandomForest   n_cal=  250 |   7.5s |  64 rows | methods=8 | train_aug=[np.False_] | NaN=0 | convergence warnings=0
RandomForest   n_cal= 1250 |   6.8s |  64 rows | methods=8 | train_aug=[np.False_] | NaN=0 | convergence warnings=0
MLP            n_cal=  250 |  11.7s |  64 rows | methods=8 | train_aug=[np.False_] | NaN=0 | convergence warnings=0
MLP            n_cal= 1250 |  11.9s |  64 rows | methods=8 | train_aug=[np.False_] | NaN=0 | convergence warnings=1

projected total at R=300: 3.2 h
   RandomForest   n_cal=  250:    37 min
   RandomForest   n_cal= 1250:    34 min
   MLP            n_cal=  250:    58 min
   MLP            n_cal= 1250:    60 min


## 4. Run

Re-runnable: finished configurations are skipped and interrupted ones resume
from their checkpoint. After a disconnect, re-run cells 1, 2 and 4.

In [ ]:
import traceback

for model, n_cal in todo:
    cfg = replace(BASE_CFG, model=model, n_cal=n_cal, n_reps=R, strict=False)
    print(f"\n=== {model}  n_cal={n_cal}  E[N_rare]={cfg.expected_rare_cal():.0f}",
          flush=True)
    try:
        run(cfg, verbose=True, checkpoint_every=10)
    except Exception:
        traceback.print_exc()
        print("--- configuration failed, continuing", flush=True)

print("\n=== pass complete")


=== RandomForest  n_cal=250  E[N_rare]=10
[s3_nuisance_RandomForest_ncal00250_a01_B16_exact] 10/300 (checkpoint)
[s3_nuisance_RandomForest_ncal00250_a01_B16_exact] 20/300 (checkpoint)
[s3_nuisance_RandomForest_ncal00250_a01_B16_exact] 30/300 (checkpoint)
[s3_nuisance_RandomForest_ncal00250_a01_B16_exact] 40/300 (checkpoint)
[s3_nuisance_RandomForest_ncal00250_a01_B16_exact] 50/300 (checkpoint)
[s3_nuisance_RandomForest_ncal00250_a01_B16_exact] 60/300 (checkpoint)
[s3_nuisance_RandomForest_ncal00250_a01_B16_exact] 70/300 (checkpoint)
[s3_nuisance_RandomForest_ncal00250_a01_B16_exact] 80/300 (checkpoint)
[s3_nuisance_RandomForest_ncal00250_a01_B16_exact] 90/300 (checkpoint)
[s3_nuisance_RandomForest_ncal00250_a01_B16_exact] 100/300 (checkpoint)
[s3_nuisance_RandomForest_ncal00250_a01_B16_exact] 110/300 (checkpoint)
[s3_nuisance_RandomForest_ncal00250_a01_B16_exact] 120/300 (checkpoint)
[s3_nuisance_RandomForest_ncal00250_a01_B16_exact] 130/300 (checkpoint)
[s3_nuisance_RandomForest_ncal

/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


[s3_nuisance_MLP_ncal00250_a01_B16_exact] 10/300 (checkpoint)


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptro

[s3_nuisance_MLP_ncal00250_a01_B16_exact] 20/300 (checkpoint)


...

[s3_nuisance_MLP_ncal01250_a01_B16_exact] 280/300 (checkpoint)


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


[s3_nuisance_MLP_ncal01250_a01_B16_exact] 290/300 (checkpoint)


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


[s3_nuisance_MLP_ncal01250_a01_B16_exact] 300/300 (checkpoint)
[done] s3_nuisance_MLP_ncal01250_a01_B16_exact  19200 rows

=== pass complete


## 5. Progress

In [ ]:
from simulate import _find, _read

rows = []
for model, n_cal in todo:
    cfg  = replace(BASE_CFG, model=model, n_cal=n_cal, n_reps=R)
    stem = os.path.join(RESULTS_DIR, f"{cfg.tag()}__long")
    final, ckpt = _find(stem), _find(stem + "__ckpt")
    if final:
        status, done = "complete", R
    elif ckpt:
        status, done = "in progress", int(_read(ckpt)["rep"].max()) + 1
    else:
        status, done = "not started", 0
    rows.append({"model": model, "n_cal": n_cal,
                 "E_rare": cfg.expected_rare_cal(),
                 "status": status, "reps_done": done, "of": R})

prog = pd.DataFrame(rows)
display(prog)
print(f"{(prog.status=='complete').sum()}/{len(prog)} complete | "
      f"{prog.reps_done.sum()}/{R*len(prog)} replications")

,model,n_cal,E_rare,status,reps_done,of
0,RandomForest,250,10.0,complete,300,300
1,RandomForest,1250,50.0,complete,300,300
2,MLP,250,10.0,complete,300,300
3,MLP,1250,50.0,complete,300,300


4/4 complete | 1200/1200 replications


## 6. Results

XGBoost rows are read from the main results directory. Absolute set sizes are
not comparable across classifiers, since they depend on predictive accuracy;
the relative within-model reduction is, and it is computed replication by
replication so that the two arms share everything except the score.

In [ ]:
import metrics as M

df_rob  = M.load(RESULTS_DIR)
df_main = M.load(MAIN_DIR)
df_main = df_main[(df_main.model == "XGBoost") & (df_main.alpha == 0.10)
                  & (df_main.n_cal.isin(NCAL)) & (~df_main.train_aug)]

df   = pd.concat([df_main, df_rob], ignore_index=True)
rep  = M.per_replication(df)
summ = M.summarize(rep)
print(f"{len(df):,} long rows | models: {sorted(df.model.unique())}")

tab = M.classifier_robustness(rep, summ, levels=NCAL,
                              models=["XGBoost"] + MODELS)
display(tab.round({"WCU APS": 3, "WCU orbit": 3, "AvgSize APS": 3,
                   "AvgSize orbit": 3, "Reduction (%)": 2, "MCSE": 2,
                   "P(delta<0)": 2, "within_orbit_var": 4, "macro_f1": 3}))
tab.to_csv(f"{RESULTS_DIR}/tab4_classifier_robustness.csv", index=False)

115,200 long rows | models: ['MLP', 'RandomForest', 'XGBoost']


,Classifier,E[N_rare],WCU APS,WCU orbit,AvgSize APS,AvgSize orbit,Reduction (%),MCSE,P(delta<0),within_orbit_var,macro_f1,n_rep
0,XGBoost,10,0.072,0.076,2.919,2.786,4.43,0.39,0.78,0.0140,0.641,300
1,XGBoost,50,0.043,0.044,2.087,1.964,5.69,0.26,0.88,0.0139,0.643,300
2,RandomForest,10,0.077,0.073,3.491,3.229,7.34,0.43,0.85,0.0074,0.566,300
3,RandomForest,50,0.042,0.045,2.607,2.363,9.16,0.24,0.99,0.0074,0.567,300
4,MLP,10,0.079,0.073,2.530,2.267,10.48,0.56,0.86,0.0351,0.679,300
5,MLP,50,0.044,0.045,1.821,1.610,11.18,0.30,0.99,0.0348,0.682,300


### Mechanism across classifiers

The theory says averaging helps to the extent that the score varies along the
orbit. If the reduction is larger where `within_orbit_var` is larger, the
account is confirmed independently of the classifier; if not, the discrepancy is
itself worth reporting.

In [ ]:
mech = tab[["Classifier", "E[N_rare]", "within_orbit_var",
            "macro_f1", "Reduction (%)", "MCSE"]].copy()
display(mech.round(4))

for n in sorted(tab["E[N_rare]"].unique()):
    s = tab[tab["E[N_rare]"] == n].sort_values("within_orbit_var")
    print(f"\nE[N_rare]={n}, ordered by within-orbit variance:")
    for _, r in s.iterrows():
        print(f"  {r.Classifier:14s} var={r.within_orbit_var:.4f}  "
              f"F1={r.macro_f1:.3f}  reduction={r['Reduction (%)']:5.2f}% "
              f"(MCSE {r.MCSE:.2f})")

,Classifier,E[N_rare],within_orbit_var,macro_f1,Reduction (%),MCSE
0,XGBoost,10,0.0140,0.6408,4.4256,0.3917
1,XGBoost,50,0.0139,0.6430,5.6928,0.2552
2,RandomForest,10,0.0074,0.5659,7.3436,0.4303
3,RandomForest,50,0.0074,0.5674,9.1550,0.2368
4,MLP,10,0.0351,0.6785,10.4765,0.5573
5,MLP,50,0.0348,0.6823,11.1838,0.2952



E[N_rare]=10, ordered by within-orbit variance:
  RandomForest   var=0.0074  F1=0.566  reduction= 7.34% (MCSE 0.43)
  XGBoost        var=0.0140  F1=0.641  reduction= 4.43% (MCSE 0.39)
  MLP            var=0.0351  F1=0.679  reduction=10.48% (MCSE 0.56)

E[N_rare]=50, ordered by within-orbit variance:
  RandomForest   var=0.0074  F1=0.567  reduction= 9.16% (MCSE 0.24)
  XGBoost        var=0.0139  F1=0.643  reduction= 5.69% (MCSE 0.26)
  MLP            var=0.0348  F1=0.682  reduction=11.18% (MCSE 0.30)


### TTA-Avg

Run in every configuration but kept out of the main robustness table, since the
question here is whether averaging helps at all rather than where it is
applied. Reported for the appendix.

In [ ]:
tta = M.classifier_robustness(rep, summ, levels=NCAL,
                              models=["XGBoost"] + MODELS,
                              treatment="classwise_ttaavg")
display(tta[["Classifier", "E[N_rare]", "AvgSize APS", "AvgSize orbit",
             "Reduction (%)", "MCSE", "P(delta<0)"]]
        .rename(columns={"AvgSize orbit": "AvgSize TTA"}).round(3))

orbit_vs_tta = pd.concat([
    M.paired_delta(rep[(rep.model == m) & (~rep.train_aug)], "AvgSize",
                   "classwise_ttaavg", "classwise_orbitavg").assign(model=m)
    for m in ["XGBoost"] + MODELS])
display(orbit_vs_tta[["model", "n_cal", "mean_delta", "mcse",
                      "frac_negative", "n"]].round(4))

,Classifier,E[N_rare],AvgSize APS,AvgSize TTA,Reduction (%),MCSE,P(delta<0)
0,XGBoost,10,2.919,2.809,3.585,0.399,0.697
1,XGBoost,50,2.087,1.981,4.855,0.263,0.877
2,RandomForest,10,3.491,3.334,4.255,0.453,0.690
3,RandomForest,50,2.607,2.465,5.250,0.275,0.887
4,MLP,10,2.530,2.251,11.141,0.579,0.877
5,MLP,50,1.821,1.592,12.209,0.304,0.990


,model,n_cal,mean_delta,mcse,frac_negative,n
0,XGBoost,250,-0.0228,0.0050,0.5800,300
1,XGBoost,1250,-0.0178,0.0023,0.6367,300
0,RandomForest,250,-0.1043,0.0085,0.7767,300
1,RandomForest,1250,-0.1017,0.0045,0.9267,300
0,MLP,250,0.0157,0.0036,0.4067,300
1,MLP,1250,0.0182,0.0022,0.3100,300


## 7. LaTeX export

In [ ]:
tex = tab[["Classifier", "E[N_rare]", "WCU APS", "WCU orbit",
           "AvgSize APS", "AvgSize orbit", "Reduction (%)", "P(delta<0)"]].copy()
tex["Reduction (%)"] = tex.apply(
    lambda r: f"{r['Reduction (%)']:.2f} ({tab.loc[r.name, 'MCSE']:.2f})", axis=1)

path = f"{RESULTS_DIR}/tab4_classifier_robustness.tex"
with open(path, "w") as fh:
    fh.write(tex.to_latex(
        index=False, escape=False, float_format="%.3f", na_rep="",
        caption=("Sensitivity of the classwise comparison to the underlying "
                 "probabilistic classifier, at two calibration regimes. "
                 "Relative reductions and paired probabilities compare the "
                 "orbit-averaged score with ordinary classwise APS within each "
                 "replication; Monte Carlo standard errors are in parentheses. "
                 "Absolute set sizes are not comparable across classifiers, "
                 "since they depend on predictive accuracy; the relative "
                 "reduction is."),
        label="tab:classifier_robustness"))
print("written:", path)
display(tex)

written: /content/drive/MyDrive/Conformal_Prediction_Research/Class_Conditional_CP_with_Data_Augmentation/csda_revision/results_robust_clf/tab4_classifier_robustness.tex


,Classifier,E[N_rare],WCU APS,WCU orbit,AvgSize APS,AvgSize orbit,Reduction (%),P(delta<0)
0,XGBoost,10,0.072000,0.076073,2.918915,2.785992,4.43 (0.39),0.776667
1,XGBoost,50,0.043147,0.044227,2.087410,1.963592,5.69 (0.26),0.883333
2,RandomForest,10,0.076633,0.072933,3.491408,3.229298,7.34 (0.43),0.850000
3,RandomForest,50,0.041760,0.045200,2.606748,2.363265,9.16 (0.24),0.986667
4,MLP,10,0.078693,0.072500,2.529758,2.266830,10.48 (0.56),0.863333
5,MLP,50,0.043593,0.044920,1.820883,1.610288,11.18 (0.30),0.990000


In [ ]:
# Diagnostic: is the ordering of the gains explained by within-orbit variation?
# `var_within_orbit` in the results measures the true-label score only. The set
# size depends on the whole score vector, so a classifier whose orbital
# variation sits on the wrong labels would not show up there. This recomputes
# the variance over all candidate labels on a handful of replications.
import numpy as np, pandas as pd
from dataclasses import replace
from simulate import (SimConfig, generate_population, split_population,
                      build_model)
from groups import SignFlipGroup
from scores import aps_scores, draw_uniform, orbit_softmax

N_PROBE = 10          # replications; this is a diagnostic, not an estimate

rows = []
for model in ["XGBoost", "RandomForest", "MLP"]:
    cfg = replace(BASE_CFG, model=model, n_cal=1250)
    for r in range(N_PROBE):
        seeds = np.random.SeedSequence([cfg.base_seed, r]).spawn(7)
        rng_data, rng_g, _, _, _, rng_utbal, rng_model = (
            np.random.default_rng(s) for s in seeds)

        X, y = generate_population(cfg, rng_data)
        _, _, _, tb = split_population(X, y, cfg, rng_data)
        X_tb, y_tb = X[tb], y[tb]

        group = SignFlipGroup(X.shape[1], cfg.n_nuisance)
        eps = group.sample(cfg.n_transforms, rng_g)
        U = draw_uniform(len(y_tb), cfg.n_classes, rng_utbal)

        m = build_model(cfg.model, seed=int(rng_model.integers(0, 2**31 - 1)))
        tr, ca, _, _ = split_population(X, y, cfg, np.random.default_rng(seeds[0]))
        m.fit(X[tr], y[tr])

        orb = orbit_softmax(m, X_tb, group, eps)
        per = np.stack([aps_scores(p, U) for p in orb], axis=0)
        idx = np.arange(len(y_tb))

        rows.append({
            "Classifier": model, "rep": r,
            "var_true_label": float(per[:, idx, y_tb].var(axis=0, ddof=0).mean()),
            "var_all_labels": float(per.var(axis=0, ddof=0).mean()),
        })

probe = pd.DataFrame(rows).groupby("Classifier")[
    ["var_true_label", "var_all_labels"]].mean()
probe["ratio"] = probe.var_all_labels / probe.var_true_label
display(probe.round(5))

/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptro

,var_true_label,var_all_labels,ratio
Classifier,,,
MLP,0.03414,0.01195,0.35015
RandomForest,0.00720,0.00465,0.64563
XGBoost,0.01300,0.00475,0.36495
